In [ ]:
import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
#Make outputs reproducible
np.random.seed(42)

#Nicer float display in pandas
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Part 1 - Data Cleaning & Preprocessing

Real datasets are almost never clean. Before training any ML model we need to handle things like:

-Missing Values (´NaN)
-Duplicated rows
-Inconsistent text (capitalization, whitespace)
-Wrong data typers
-Outliers
-Features on very different scales

We will build three small synthetic datasets, each designed to ilustrate a especific problem

In [ ]:
dataset_a = pd.DataFrame({
    "student_id": [1,2,3,4,5,6,7,8],
    "age": [20,21,np.nan,22,23,20,np.nan,24],
    "study_hours": [10,12,8,np.nan,15,9,11,14],
    "score": [78,82,65,70,np.nan,72,80,88]
})

dataset_a

In [ ]:
#Step 1: Detect missing values per column
print("Missing values per column: ")
print(dataset_a.isna().sum())

In [ ]:
#Step 2: impute (fill) missing values
#Strategy: fill mnumeric columns with the column mean.
#This is a simple, very common baseline strategy.

dataset_a_clean = dataset_a.copy()

for col in ["age", "study_hours", "score"]:
    mean_value = dataset_a_clean[col].mean()
    dataset_a_clean[col] = dataset_a_clean[col].fillna(mean_value)
dataset_a_clean

**Note:** The mean is only one of several imputation strategies. Other common options are the **median** (more robust to outliers), the **mode** (useful for categorical columns), or more advance techniques like KNN imputation. The right choice depends on the data.

In [ ]:
dataset_b = pd.DataFrame({
    "brand": ["Toyota", "toyota", "Honda", "Ford", "ford", "Toyota", "Honda ", "TOYOTA"],
    "model": ["Corolla", "Corolla", "Civic", "Focus", "Focus", "Corolla", "Civic ", "Corolla"],
    "price": ["15000", "15000", "18000", "12000", "12000", "15000", "18500 ", "15000"],
    "year": [2018,2018,2019,2017,2017,2018,2020,2018]
})

dataset_b

In [ ]:
#Step 1: check data types
#Notice that "price" is an 'object' (string), not a number.

dataset_b.dtypes

In [ ]:
# Step 2: normalize text in the 'brand' column
# - strip() removes leading/trailing whitespace
# - str.title() converts to Title Case ("toyota" -> "Toyota")

dataset_b_clean = dataset_b.copy()
dataset_b_clean["brand"] = dataset_b_clean["brand"].str.strip().str.title()

In [ ]:
#Step 3: convert 'price' from string to numeric
dataset_b_clean["proce"] = pd.to_numeric(dataset_b_clean["price"])

In [ ]:
#Step 4: remove duplicated rows
print(f"Rows before removing duplicates: {len(dataset_b_clean)}")
dataset_b_clean = dataset_b_clean.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(dataset_b_clean)}")

dataset_b_clean

### Dataset C - Outliers and Features Scaling

Outliers are values that are very different from the rest of the data.
They can be legitimate, but they can also be errors (e.g., a typo).

Feature scaling is important when features live on very different numerical ranges - for example, a car's weight (thousands of pounds) vs . its number of cylinders (4-8). Without scaling, features with larger ranges can dominate the learning process. 

In [ ]:
dataset_c = pd.DataFrame({
    "weight": [3.50,3.69,3.44,3.43,4.34,4.42,2.37,25.00],
    "horsepower": [130,165,150,140,198,220,95,180],
    "cylynders": [4,6,4,4,8,8,4,6],
    "mpg": [18,15,18,16,15,14,24,17]
})

dataset_c

In [ ]:
#Step 1: detect outliers in 'weight' using the IQR (InterQuartile Range) rule
# A value is considered an outlier if it falls outside [01 - 1.5*IQR, Q3 + 1.5*IQR].
q1 = dataset_c["weight"].quantile(0.25)
q3 = dataset_c["weight"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"Valid range for 'weight': [{lower_bound:.2f}, {upper_bound:.2f}]")

outliers_mask = (dataset_c["weight"] < lower_bound) | (dataset_c["weight"] > upper_bound)

print("\nOutlier rows:")
print(dataset_c[outliers_mask])